<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 2 — Su primer motor de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">TF-IDF + similitud coseno, implementados desde cero · sin librerías de NLP</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.

## Objetivo

Dado el corpus que preprocesaron en el **Lab 1**, responder consultas en lenguaje libre devolviendo un **ranking por relevancia**. La regla del curso aplica: los algoritmos núcleo (TF, IDF, TF-IDF y coseno) se implementan **desde cero**. Está prohibido `TfidfVectorizer` u otra caja negra para resolver el ejercicio (solo se permite, opcionalmente, al final para *verificar*).

## 0 · Cargar el corpus procesado del Lab 1

Debe estar `corpus_procesado.json` en la misma carpeta. Si no, vuelvan a correr el Lab 1.

In [1]:
import json, math

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)

documentos = [d['tokens'] for d in corpus]          # lista de listas de terminos
print(f'{len(corpus)} documentos.  Ejemplo {corpus[0]["id"]}:', documentos[0][:8])

14 documentos.  Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


**Reutilicen su `preprocesar`.** Péguenla aquí *idéntica* a la del Lab 1. Es indispensable que la consulta pase por **exactamente el mismo pipeline** que el corpus (este es el error más común: vectorizar la consulta distinto y que nada coincida).

In [2]:
import re, unicodedata
from nltk.stem import SnowballStemmer
import spacy

try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

# Regex para detección de ruido (idéntico a Lab 1)
RE_URL = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_EMOJI = re.compile(r'[\U0001F300-\U0001F9FF]|[\U0001F600-\U0001F64F]|[\U0001F680-\U0001F6FF]|[😀-🤯🦀-🦟]', flags=re.UNICODE)

# Stopwords personalizados (idéntico a Lab 1)
stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'muy', 'mas'}
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

# Stemmer (idéntico a Lab 1)
stemmer = SnowballStemmer('spanish')

def normalizar(texto):
    """Idéntico a Lab 1"""
    texto = texto.lower()
    texto_nfd = unicodedata.normalize('NFD', texto)
    texto_sin_acentos = ''.join(c for c in texto_nfd if unicodedata.category(c) != 'Mn')
    texto_sin_acentos = RE_URL.sub('', texto_sin_acentos)
    texto_sin_acentos = RE_HTML.sub('', texto_sin_acentos)
    texto_sin_acentos = RE_EMOJI.sub('', texto_sin_acentos)
    texto_sin_acentos = re.sub(r'#\s*', '', texto_sin_acentos)
    texto_sin_acentos = re.sub(r'\s+', ' ', texto_sin_acentos).strip()
    return texto_sin_acentos

def preprocesar(texto):
    """Idéntico a Lab 1: normalizar + lemmatización"""
    texto_norm = normalizar(texto)
    doc = nlp(texto_norm)
    tokens_lemma_list = [token.lemma_ for token in doc if token.lemma_ not in MIS_STOPWORDS and token.is_alpha]
    return tokens_lemma_list

# Test
print("preprocesar OK:", preprocesar('Problemas de agua potable'))

preprocesar OK: ['problema', 'agua', 'potable']


## 1 · Indexar — TF, IDF y TF-IDF desde cero

Implementen las tres funciones siguiendo las fórmulas de la clase. La estructura es la del slide de "15 líneas".
- `tf(doc)` → frecuencia **relativa**: `f(t,d) / |d|` (importancia local).
- `idf(corpus)` → `ln(N / df(t))` (rareza global). Cuenten **documentos**, no apariciones.
- `tfidf(doc, idf_)` → `tf(t,d) · idf(t)`.

In [3]:
def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    # Contar frecuencia de cada término
    freq = {}
    for term in doc:
        freq[term] = freq.get(term, 0) + 1
    
    # Normalizar por longitud del documento
    doc_len = len(doc)
    tf_dict = {}
    for term, count in freq.items():
        tf_dict[term] = count / doc_len
    
    return tf_dict

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf.
    idf = ln(N / df(t)) donde df(t) = número de documentos que contienen t
    """
    N = len(corpus)  # Total de documentos
    
    # Contar en cuántos documentos aparece cada término
    doc_freq = {}
    for doc in corpus:
        unique_terms = set(doc)  # Usar set para contar documentos, no frecuencias
        for term in unique_terms:
            doc_freq[term] = doc_freq.get(term, 0) + 1
    
    # Calcular IDF: ln(N / df(t))
    idf_dict = {}
    for term, df_t in doc_freq.items():
        idf_dict[term] = math.log(N / df_t)
    
    return idf_dict

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento.
    Terminos no vistos en idf_ -> peso 0
    """
    tf_dict = tf(doc)
    tfidf_dict = {}
    
    for term, tf_value in tf_dict.items():
        # Si el término está en el IDF del corpus, multiplicar; si no, peso 0
        if term in idf_:
            tfidf_dict[term] = tf_value * idf_[term]
        else:
            tfidf_dict[term] = 0
    
    return tfidf_dict

In [4]:
# Construir el indice: un vector tf-idf (dict) por documento
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check: el termino mas pesado de d04 (sequia/maiz) deberia ser tematico
import operator
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Terminos top de', corpus[3]['id'], '->', top)

Terminos top de d04 -> [('gravemente', 0.1759371553076839), ('cultivo', 0.1759371553076839), ('maiz', 0.1759371553076839), ('frijol', 0.1759371553076839), ('fronterizo', 0.1759371553076839)]


## 2 · Procesar la consulta

La consulta es texto libre. Pásenla por el **mismo** `preprocesar`, conviértanla en vector con el **mismo** `IDF` del corpus (no recalculen IDF con la consulta dentro).

In [5]:
def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF del corpus."""
    # Paso 1: Preprocesar la consulta
    tokens = preprocesar(texto)
    
    # Paso 2: Calcular TF de la consulta
    tf_consulta = tf(tokens)
    
    # Paso 3: Multiplicar por IDF del corpus (no recalcular IDF)
    tfidf_consulta = {}
    for term, tf_value in tf_consulta.items():
        if term in IDF:
            tfidf_consulta[term] = tf_value * IDF[term]
        else:
            # Término no visto en corpus: peso 0
            tfidf_consulta[term] = 0
    
    return tfidf_consulta

# Test
q = vectorizar_consulta('sequia en los cultivos')
print("Consulta vectorizada:", {k: v for k, v in sorted(q.items(), key=lambda x: -x[1])[:5]})

Consulta vectorizada: {'cultivo': 1.3195286648076292, 'sequia': 0.9729550745276566}


## 3 · Ranquear — similitud coseno

Implementen el coseno entre dos vectores dispersos (dicts) y la función `buscar` que devuelve el **top-k**.
$$\cos(\vec{q},\vec{d}) = \frac{\vec{q}\cdot\vec{d}}{\lVert\vec{q}\rVert\,\lVert\vec{d}\rVert}$$

In [6]:
def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    # Producto punto: sumar productos de términos comunes
    dot_product = 0
    for term in v1:
        if term in v2:
            dot_product += v1[term] * v2[term]
    
    # Norma de v1: sqrt(suma de cuadrados)
    norm_v1 = math.sqrt(sum(x**2 for x in v1.values()))
    
    # Norma de v2
    norm_v2 = math.sqrt(sum(x**2 for x in v2.values()))
    
    # Coseno: dot_product / (norm_v1 * norm_v2)
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0  # Si alguno es vector nulo
    
    return dot_product / (norm_v1 * norm_v2)

def buscar(consulta, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(consulta)
    
    # Calcular similitud coseno contra cada documento
    scores = []
    for i, doc_vector in enumerate(INDICE):
        score = coseno(q, doc_vector)
        scores.append((corpus[i]['id'], corpus[i]['titulo'], score))
    
    # Ordenar por score descendente y retornar top-k
    scores_sorted = sorted(scores, key=lambda x: -x[2])[:k]
    
    return scores_sorted

In [7]:
# Prueba: deberia rankear alto d02 (sequia) y d04 (sequia/maiz)
print('Consulta: "sequia y cultivos de maiz"\n')
for id_, titulo, score in buscar('sequia y cultivos de maiz'):
    print(f'{score:.3f}  {id_}  {titulo}')

Consulta: "sequia y cultivos de maiz"

0.447  d04  Sequia afecta cultivos de maiz
0.083  d02  Crisis hidrica golpea la region
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d05  Turismo crece en el Canon del Sumidero


## 4 · Rómpanlo

Entender dónde falla un modelo vale más que celebrarlo donde funciona. Empezamos con el caso de clase:

In [8]:
print('Consulta: "problemas de agua"\n')
for id_, titulo, score in buscar('problemas de agua'):
    print(f'{score:.3f}  {id_}  {titulo}')

print('\n--- OBSERVACIÓN ---')
print('d13 (agua potable) sale bien, pero d02 (crisis HIDRICA) — claramente relevante —')
print('queda hundido o en 0. "agua" e "hidrico" son cadenas distintas: TF-IDF no sabe que son sinonimos.')

Consulta: "problemas de agua"

0.476  d13  Restablecen servicio de agua potable
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz

--- OBSERVACIÓN ---
d13 (agua potable) sale bien, pero d02 (crisis HIDRICA) — claramente relevante —
queda hundido o en 0. "agua" e "hidrico" son cadenas distintas: TF-IDF no sabe que son sinonimos.


**4.a** Encuentren **2 consultas propias** donde su buscador devuelva resultados malos. Para cada una: muestren la salida y **expliquen la causa** (sinonimia, polisemia, falta de contexto, preprocesamiento agresivo, etc.). Traigan estas dos consultas a la próxima clase: con ellas abriremos BM25.

In [9]:
# CONSULTA FALLIDA 1: Sinonimia - 'turismo' vs 'viajeros'
print('CONSULTA FALLIDA 1: "turismo y viajeros"\n')
for id_, titulo, score in buscar('turismo viajeros destinos'):
    print(f'{score:.3f}  {id_}  {titulo}')

CONSULTA FALLIDA 1: "turismo y viajeros"

0.364  d09  San Cristobal, destino cultural
0.000  d01  Lluvias provocan inundaciones en Tuxtla
0.000  d02  Crisis hidrica golpea la region
0.000  d03  Cafe de Chiapas rompe record de exportacion
0.000  d04  Sequia afecta cultivos de maiz


_Causa de la falla 1:_ **Sinonimia** (falta de cobertura semántica). La consulta contiene 'turismo' y 'viajeros', pero d09 ('San Cristobal, destino cultural') usa 'viajeros' y 'destino cultural' sin usar la palabra 'turismo'. TF-IDF es ciego a que 'viajeros' es un sinónimo o concepto relacionado. Aunque 'viajeros' aparece explícitamente en d09, el motor busca match exacto de términos y no entiende relaciones semánticas.

In [10]:
# CONSULTA FALLIDA 2: Polisemia - 'mercado' (comercio vs estructura económica)
print('\nCONSULTA FALLIDA 2: "mercados agricolas productores"\n')
for id_, titulo, score in buscar('mercados agricolas de productores'):
    print(f'{score:.3f}  {id_}  {titulo}')


CONSULTA FALLIDA 2: "mercados agricolas productores"

0.141  d08  Repunta la produccion de cacao
0.140  d03  Cafe de Chiapas rompe record de exportacion
0.140  d12  Feria celebra el cafe y el cacao
0.134  d09  San Cristobal, destino cultural
0.000  d01  Lluvias provocan inundaciones en Tuxtla


_Causa de la falla 2:_ **Polisemia + contexto débil**. La consulta 'mercados agricolas' puede referirse a (1) mercados físicos donde se vende, o (2) mercados económicos. d09 contiene 'mercados' refiriéndose a mercados físicos (parte del turismo cultural en San Cristóbal), mientras que d12 contiene 'mercados premium' refiriéndose a segmento de comprador. La consulta es ambigua y TF-IDF no tiene contexto para desambiguar. Además, 'd12 (feria)' es más relevante por tema (café/cacao/productores) pero 'd09' rankea alto solo porque contiene 'mercados'. La falta de análisis sintáctico y de desambiguación hace que el motor no entienda que 'mercados' en el contexto de productores agrícolas se refiere a destinatarios finales, no a ubicaciones.

## 5 · (Opcional) Verificación contra scikit-learn

Solo para **comprobar** que su implementación desde cero es correcta — no sustituye al ejercicio ni cuenta como entrega. Si sus rankings coinciden a grandes rasgos con los de `TfidfVectorizer`, van bien.

In [11]:
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity
# docs_txt = [' '.join(t) for t in documentos]
# vec = TfidfVectorizer()
# X = vec.fit_transform(docs_txt)
# q = vec.transform([' '.join(preprocesar('sequia y cultivos de maiz'))])
# sims = cosine_similarity(q, X)[0]
# print(sorted(zip(sims, [d['id'] for d in corpus]), reverse=True)[:5])

## Entregables del Lab 2

- [x] `tf`, `idf`, `tfidf`, `coseno`, `vectorizar_consulta` y `buscar` implementadas desde cero.
- [x] El índice construido y la prueba de `buscar` ejecutada con salida.
- [x] La demostración de la falla "problemas de agua" vs. "crisis hídrica" ejecutada.
- [x] **2 consultas fallidas propias** con su causa explicada por escrito.
- [x] `AI_USAGE.md` actualizado si usaron IA.